In [4]:
!pip install pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.9 MB 13.2 MB/s eta 0:00:01
   ----------------------- ---------------- 5.8/9.9 MB 14.8 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 16.5 MB/s  0:00:00

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   ---------------

In [2]:
# load libraries 
import pandas as pd 

In [3]:
# load data 
data_all = pd.read_csv('data_mutations.txt', sep='\t') # first row is header
data_all.head()

,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,...,t_alt_count,n_ref_count,n_alt_count,HGVSc,HGVSp,HGVSp_Short,Transcript_ID,RefSeq,Protein_position,Codons
0,ARID1A,8289,MSKCC,GRCh37,1,27101374,27101374,+,missense_variant,Missense_Mutation,...,195,524,0,ENST00000324856.7:c.4656G>T,p.Gln1552His,p.Q1552H,ENST00000324856,NM_006015.4,1552.0,caG/caT
1,CSF3R,1441,MSKCC,GRCh37,1,36933165,36933165,+,missense_variant,Missense_Mutation,...,106,739,0,ENST00000361632.4:c.1952G>T,p.Ser651Ile,p.S651I,ENST00000361632,NaN,651.0,aGc/aTc
2,MUTYH,4595,MSKCC,GRCh37,1,45797106,45797106,+,missense_variant,Missense_Mutation,...,522,821,1,ENST00000372115.3:c.1267C>T,p.Arg423Trp,p.R423W,ENST00000372115,NM_001048171.1,423.0,Cgg/Tgg
3,PIK3R3,8503,MSKCC,GRCh37,1,46509362,46509362,+,missense_variant,Missense_Mutation,...,227,644,2,ENST00000262741.5:c.1369C>A,p.Pro457Thr,p.P457T,ENST00000262741,NM_003629.3,457.0,Ccc/Acc
4,RAD54L,8438,MSKCC,GRCh37,1,46743563,46743563,+,missense_variant,Missense_Mutation,...,102,697,0,ENST00000371975.4:c.1944G>C,p.Glu648Asp,p.E648D,ENST00000371975,NM_003579.3,648.0,gaG/gaC


In [4]:
# what are all the column names?
headers = data_all.columns.tolist()
print(headers)

['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Consequence', 'Variant_Classification', 'Variant_Type', 'Reference_Allele', 'Tumor_Seq_Allele1', 'Tumor_Seq_Allele2', 'dbSNP_RS', 'dbSNP_Val_Status', 'Tumor_Sample_Barcode', 'Matched_Norm_Sample_Barcode', 'Match_Norm_Seq_Allele1', 'Match_Norm_Seq_Allele2', 'Tumor_Validation_Allele1', 'Tumor_Validation_Allele2', 'Match_Norm_Validation_Allele1', 'Match_Norm_Validation_Allele2', 'Verification_Status', 'Validation_Status', 'Mutation_Status', 'Sequencing_Phase', 'Sequence_Source', 'Validation_Method', 'Score', 'BAM_File', 'Sequencer', 't_ref_count', 't_alt_count', 'n_ref_count', 'n_alt_count', 'HGVSc', 'HGVSp', 'HGVSp_Short', 'Transcript_ID', 'RefSeq', 'Protein_position', 'Codons']


**Preprocessing and Making the Mutation Matrix**

In [ ]:
# proposed gene list 
gene_list = [
    # Main genes 
    "TP53", "RB1", "ATM",

    # RTK/RAS/PI3K pathway 
    "KRAS", "EGFR", "PIK3CA", "AKT3", "PTEN", "ERBB4",

    # Wnt pathway 
    "APC", "CTNNB1",

    # Lung Adenocarcinoma
    "STK11", "SMARCA4", "PBRM1", "CREBBP",

    # Extras
    "MGA", "PTPRD"
]

In [ ]:
# determine different types of mutations 
unique_values = data_all['Variant_Classification'].unique()

# list coding mutations 
coding_mutations = [
    "Missense_Mutation", "Nonsense_Mutation",
    "Frame_Shift_Ins", "Frame_Shift_Del",
    "Splice_Site"
]

# filter for only coding mutations 
data_filtered = data_all[data_all["Variant_Classification"].isin(coding_mutations)]

# filter for only genes in the gene list 
data_filtered = data_filtered[data_filtered["Hugo_Symbol"].isin(gene_list)]


# filter for only Lung Adenocarcinoma patients by joining another table with only those cancer types 
clinical_data_all = pd.read_csv('lung_msk_mind_2020_clinical_data (1).tsv', sep='\t')
clinical_data_LA = clinical_data_all[(clinical_data_all["Cancer Type"] == "Lung Adenocarcinoma") & (clinical_data_all["Cancer Type Detailed"] == "Lung Adenocarcinoma")].copy()

# verify tables can be merged/that they use the same tumor IDs
print(clinical_data_LA['Sample ID'])
print(data_filtered['Tumor_Sample_Barcode'])

# merge tables 
df = pd.merge(
    data_filtered,
    clinical_data_LA,
    left_on='Tumor_Sample_Barcode',
    right_on='Sample ID',
    how='inner'
)

# pivot table: one row per sample, one column per gene
# 1 = mutation at gene; 0 = no mutation at gene 
mutation_matrix = (
    df.assign(mut=1)
      .pivot_table(
          index="Tumor_Sample_Barcode",
          columns="Hugo_Symbol",
          values="mut",
          aggfunc="max",
          fill_value=0
      )
)

0     P-0002794-T01-IM3
1     P-0005783-T02-IM6
2     P-0006713-T01-IM5
3     P-0018763-T01-IM6
4     P-0018960-T01-IM6
5     P-0019098-T01-IM6
6     P-0020140-T02-IM6
7     P-0020422-T01-IM6
8     P-0021266-T01-IM6
9     P-0022001-T01-IM6
10    P-0023926-T01-IM6
11    P-0025917-T02-IM6
12    P-0027733-T01-IM6
13    P-0028723-T01-IM6
14    P-0033979-T01-IM6
15    P-0044186-T01-IM6
16    P-0045298-T01-IM6
Name: Sample ID, dtype: str
10      P-0005783-T02-IM6
15      P-0005783-T02-IM6
16      P-0005783-T02-IM6
20      P-0005783-T02-IM6
26      P-0005783-T02-IM6
              ...        
2940    P-0018960-T01-IM6
2942    P-0018960-T01-IM6
2945    P-0018960-T01-IM6
2946    P-0018960-T01-IM6
2947    P-0018960-T01-IM6
Name: Tumor_Sample_Barcode, Length: 575, dtype: str


Sanity Checks after preprocessing

In [29]:
# verify process of joining tables 
print(df.shape) # (55, 122)
print(df["Tumor_Sample_Barcode"].nunique()) # 12
print(df["Sample ID"].nunique()) # 12 
# there are only 12 rows/patients instead of 17 b/c patients were excluded for not having coding-mutations
# there are 55 rows instead of 12 because each row = one mutation event, and each sample has multiple mutations


# make sure all LUAD samples are kept, including samples with 0 retained mutations
all_luad_samples = clinical_data_LA["Sample ID"].unique()
mutation_matrix = mutation_matrix.reindex(all_luad_samples, fill_value=0)

# sanity checks
print(mutation_matrix.shape) # (17, 15)
# 17 rows = all patients; 15 columnds = all genes retained 

print(mutation_matrix.head())
# shows that the table has rows of tumor samples and columns of gene names 

# all rows' IDs are unique
print(mutation_matrix.index.is_unique) # True

# check for no NaNs 
print(sorted(pd.unique(mutation_matrix.values.ravel())))

# check for samples with at least one retained mutation
print((mutation_matrix.sum(axis=1) > 0).sum()) # 12
# means there are 5 tumor samples with no coding/non-silent mutations in selected genes

# check which genes were retained after joining tables
print(mutation_matrix.columns.tolist())
# lost two genes: PTEN (RTK/RAS/PI3K pathway) and CTNNB1 (Wnt pathway)

# for each sample-gene pair in the merged long table, the matrix entry should be 1
check_pairs = df[["Tumor_Sample_Barcode", "Hugo_Symbol"]].drop_duplicates()

all_checks_pass = True
for _, row in check_pairs.iterrows():
    sample = row["Tumor_Sample_Barcode"]
    gene = row["Hugo_Symbol"]
    if mutation_matrix.loc[sample, gene] != 1:
        all_checks_pass = False
        print(f"Mismatch found: sample={sample}, gene={gene}")

print(all_checks_pass) # True
# means that for every mutation event in the raw data correctly appears as a 1 in the matrix
# confirms no data loss/pivot errors 

(55, 122)
12
12
(17, 15)
Hugo_Symbol           AKT3  APC  ATM  CREBBP  EGFR  ERBB4  KRAS  MGA  PBRM1  \
Tumor_Sample_Barcode                                                          
P-0002794-T01-IM3        0    0    0       0     0      0     0    0      0   
P-0005783-T02-IM6        1    1    1       1     0      1     1    1      1   
P-0006713-T01-IM5        1    0    0       0     1      0     0    1      0   
P-0018763-T01-IM6        0    0    0       0     0      0     0    0      0   
P-0018960-T01-IM6        0    0    0       1     0      1     1    0      0   

Hugo_Symbol           PIK3CA  PTPRD  RB1  SMARCA4  STK11  TP53  
Tumor_Sample_Barcode                                            
P-0002794-T01-IM3          0      0    0        0      0     0  
P-0005783-T02-IM6          0      1    1        1      0     1  
P-0006713-T01-IM5          0      1    0        0      0     1  
P-0018763-T01-IM6          0      0    0        0      0     0  
P-0018960-T01-IM6          0   

**Define binary TP53 status**

In [ ]:
mutation_matrix["TP53_status"] = mutation_matrix["TP53"].apply(
    lambda x: "Mut" if x == 1 else "WT"
)

print("TP53 status counts:")
print(mutation_matrix["TP53_status"].value_counts())

# split by TP53 status
tp53_mut = mutation_matrix[mutation_matrix["TP53_status"] == "Mut"].copy()
tp53_wt  = mutation_matrix[mutation_matrix["TP53_status"] == "WT"].copy()

TP53 status counts:
TP53_status
WT     9
Mut    8
Name: count, dtype: int64

TP53-mutant samples: 8
TP53-WT samples: 9
